# RDT-1B reproduction check (~40 min on an A100)

The pilot got **17% success on PickCube** where the RDT paper reports **77.2%**, so attribution results are not
trustworthy until the policy reproduces. This notebook finds out why by running PickCube (20 episodes, official
seeds) under five configurations:

| # | Simulator | Code | Tests |
|---|---|---|---|
| 1 | current: ManiSkill 3.0.1, SAPIEN 3.0.3, Python 3.13 | **RDT's official eval script** | Is the simulator version the problem? |
| 2 | current | this repo, fp32 (as in the pilot) | baseline of the pilot |
| 3 | current | this repo, bf16 + official `obs_mode="rgb"` | do precision / obs mode matter? |
| 4 | **RDT-era:** ManiSkill 3.0.0b15 + SAPIEN 3.0.0b1 (Nov 2024), Python 3.11 | **RDT's official eval script** | does the old simulator reproduce 77%? |
| 5 | RDT-era | this repo, fp32 | does our pipeline match the official one there? |

`Runtime → Run all`, then send the final table back.

In [ ]:
#@title 1. Setup (same as the pilot notebook)
import os, sys, subprocess, importlib
OUT_DRIVE = '/content/drive/MyDrive/rdt-xai/repro'  #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('/content/rdt-igtesting'):
    subprocess.run('git clone -q -b cvpr-xai https://github.com/rdgbrandon/rdt-igtesting /content/rdt-igtesting',
                   shell=True, check=True)
subprocess.run('cd /content/rdt-igtesting && git fetch -q origin cvpr-xai && git checkout -q cvpr-xai && git pull -q origin cvpr-xai',
               shell=True, check=True)
sys.path.insert(0, '/content/rdt-igtesting')
import rdt_xai.colab as colab
importlib.reload(colab)
OUT = colab.setup(OUT_DRIVE)

In [ ]:
#@title 2. Download models (skipped if already present)
import shutil
from huggingface_hub import hf_hub_download
D = '/content/rdt-data'
for sub in ('rdt', 'siglip', 'lang'):
    os.makedirs(f'{D}/{sub}', exist_ok=True)
for f in ['config.json', 'preprocessor_config.json', 'special_tokens_map.json', 'spiece.model',
          'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']:
    hf_hub_download('google/siglip-so400m-patch14-384', f, local_dir=f'{D}/siglip')
hf_hub_download('robotics-diffusion-transformer/maniskill-model', 'rdt/mp_rank_00_model_states.pt', local_dir=D)
for t in ['PickCube', 'StackCube', 'PushCube', 'PegInsertionSide', 'PlugCharger']:
    p = hf_hub_download('robotics-diffusion-transformer/maniskill-model', f'lang_embeds/text_embed_{t}-v1.pt', local_dir=D)
    if not os.path.exists(f'{D}/lang/text_embed_{t}-v1.pt'):
        shutil.copy(p, f'{D}/lang/text_embed_{t}-v1.pt')
print('models ready')

In [ ]:
#@title 3. Runs 1-3 in the current environment (~15 min)
import json
TASK = 'PickCube-v1'  #@param ["PickCube-v1", "StackCube-v1", "PushCube-v1"]
N = 20                #@param {type:"integer"}
ENV = dict(os.environ, RDT_REPO='/content/RoboticsDiffusionTransformer', RDT_DATA='/content/rdt-data')
results = {}


def run(tag, python, extra):
    path = f'{OUT}/repro_{tag}_{TASK}_{N}.json'
    if os.path.exists(path):
        results[tag] = json.load(open(path)); print(tag, '(cached)', results[tag]['success_rate']); return
    cmd = [python, 'tests/repro.py', '--task', TASK, '-n', str(N)] + extra
    print('>>', tag, ' '.join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd='/content/rdt-igtesting', env=ENV, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    last_err = []
    for line in p.stderr:
        if line.startswith('  ep'):
            print(line.rstrip(), flush=True)
        else:
            last_err = (last_err + [line])[-30:]
    out = p.stdout.read().strip().splitlines()
    p.wait()
    if p.returncode != 0 or not out:
        print(f'{tag} FAILED (exit {p.returncode}):\n' + ''.join(last_err)); results[tag] = None; return
    results[tag] = json.loads(out[-1])
    json.dump(results[tag], open(path, 'w'))
    print(f"{tag}: success {results[tag]['success_rate']:.0%}  ({results[tag]['minutes']:.1f} min)\n", flush=True)


run('1_official_current', sys.executable, ['--impl', 'official', '--dtype', 'bf16'])
run('2_ours_fp32_current', sys.executable, ['--impl', 'ours', '--dtype', 'fp32', '--obs', 'state_dict'])
run('3_ours_bf16_rgb_current', sys.executable, ['--impl', 'ours', '--dtype', 'bf16', '--obs', 'rgb'])
colab.sync(OUT, OUT_DRIVE)

In [ ]:
#@title 4. Build the RDT-era environment (Python 3.11, ManiSkill 3.0.0b15, SAPIEN 3.0.0b1) (~5 min)
PY311 = '/content/py311/bin/python'
if not os.path.exists(PY311):
    colab.sh('pip install -q uv')
    colab.sh('uv venv -q --python 3.11 /content/py311')
    colab.sh(f'uv pip install -q --python {PY311} "torch==2.5.1" "torchvision==0.20.1" '
             '--index-url https://download.pytorch.org/whl/cu121')
    colab.sh(f'uv pip install -q --python {PY311} "mani-skill==3.0.0b15" "sapien==3.0.0b1" "transformers==4.41.0" '
             '"diffusers==0.27.2" "timm==1.0.3" "huggingface_hub==0.23.4" "numpy<2" einops sentencepiece pyyaml '
             'imageio scipy')
print(colab.sh(f'{PY311} -c "import torch, mani_skill, sapien; print(torch.__version__, mani_skill.__version__, '
               f'sapien.__version__, torch.cuda.is_available())"'))

In [ ]:
#@title 5. Runs 4-5 in the RDT-era environment (~15 min)
run('4_official_rdt_era', PY311, ['--impl', 'official', '--dtype', 'bf16'])
run('5_ours_fp32_rdt_era', PY311, ['--impl', 'ours', '--dtype', 'fp32', '--obs', 'state_dict'])
colab.sync(OUT, OUT_DRIVE)

In [ ]:
#@title 6. Summary - send this table back
import pandas as pd
rows = []
for tag, r in results.items():
    if r is None:
        rows.append(dict(run=tag, success='FAILED')); continue
    v = r['versions']
    rows.append(dict(run=tag, success=f"{r['success_rate']:.0%}", n=r['n'], mean_steps=round(r['mean_steps']),
                     python=v['python'], mani_skill=v['mani_skill'], sapien=v['sapien'], torch=v['torch'],
                     transformers=v['transformers'], diffusers=v['diffusers'], timm=v['timm']))
df = pd.DataFrame(rows)
print('RDT paper, PickCube-v1: 77.2%')
print(df.to_string(index=False))

**How to read it**

* **4 ≈ 77% but 1 is low** → the newer simulator changed something RDT depends on; run the study in the RDT-era
  environment (the pilot notebook will get a switch for it).
* **4 and 5 both ≈ 77%** → our pipeline is faithful in that environment; go straight to the study there.
* **1 ≈ 77% but 2/3 low** → a difference in this repo's pipeline (3 vs 2 tells whether it is precision / obs mode).
* **everything low** → the checkpoint / seeds / protocol itself; send the table and the per-episode output.